In [1]:
# step1 : install and import libraries

!pip install kaggle -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
# Sklearn — allowed ONLY for evaluation & baseline models
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             precision_score, recall_score, f1_score,
                             classification_report, ConfusionMatrixDisplay)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
print(' All libraries imported successfully!')

 All libraries imported successfully!


In [2]:
#step2: load dataset

df= pd.read_csv("UCI_Credit_Card.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'UCI_Credit_Card.csv'

In [ ]:
# task 1 : data preprocessing
df.shape

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
if 'ID' in df.columns:
    df = df.drop('ID', axis=1)
else:
    print("'ID' column not found in DataFrame, skipping drop operation.")

In [ ]:
df = df.rename(columns={'default.payment.next.month': 'default'})

In [ ]:
print(df.columns)
df.head()

In [ ]:
print("Missing values per column:\n")
print(df.isnull().sum())

In [ ]:
#fixing hidden values
# Fix EDUCATION (valid: 1–4)
df['EDUCATION'] = df['EDUCATION'].replace([0, 5, 6], 4)

# Fix MARRIAGE (valid: 1–3)
df['MARRIAGE'] = df['MARRIAGE'].replace(0, 3)

In [ ]:
df.fillna(df.median(numeric_only=True), inplace=True)

In [ ]:
# we are Identifying categorical-like columns
categorical_cols = ['SEX', 'EDUCATION', 'MARRIAGE']
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)


In [ ]:
#1.4 Feature / Target Split & Normalisation

In [ ]:
# Features (everything except target)
X = df.drop('default', axis=1).values.astype(float)

# Target
y = df['default'].values.reshape(-1, 1)

In [ ]:
from sklearn.model_selection import train_test_split

# Assuming X and y are already defined from previous steps
# Features (everything except target)
# X = df.drop('default', axis=1).values.astype(float)
# Target
# y = df['default'].values.reshape(-1, 1)

# Step 1: Split into Train (70%) and Temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Step 2: Split Temp into Validation (15%) and Test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

# Compute mean and std
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

# Prevent division by zero
std[std == 0] = 1

# Apply normalization
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std
X_val = (X_val - mean) / std

In [ ]:
print("Mean (approx 0):", np.mean(X_train, axis=0)[:5])
print("Std (approx 1):", np.std(X_train, axis=0)[:5])

In [ ]:
from sklearn.model_selection import train_test_split

# Step 1: Split into Train (70%) and Temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
# Step 2: Split Temp into Validation (15%) and Test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

In [ ]:
print("Train size:", X_train.shape)
print("Validation size:", X_val.shape)
print("Test size:", X_test.shape)

In [ ]:
#TASK2: ANN Implementation(only NumPy)

In [ ]:
#activation functions
def sigmoid(Z):
    return 1 / (1 + np.exp(-Z))

def sigmoid_derivative(A):
    return A * (1 - A)

def relu(Z):
    return np.maximum(0, Z)

def relu_derivative(Z):
    return (Z > 0).astype(float)

In [ ]:
# He Initialization
def init_params(layer_dims):
    weights = []
    biases = []

    for i in range(len(layer_dims)-1):
        W = np.random.randn(layer_dims[i], layer_dims[i+1]) * np.sqrt(2 / layer_dims[i])
        b = np.zeros((1, layer_dims[i+1]))

        weights.append(W)
        biases.append(b)

    return weights, biases

In [ ]:
#dropout
def apply_dropout(A, rate):
    mask = (np.random.rand(*A.shape) > rate).astype(float)
    A = A * mask
    A = A / (1 - rate)
    return A, mask

In [ ]:
#forward propagation
def forward(X, weights, biases, dropout_rate=0.0):
    A = X
    activations = [A]
    Zs = []
    dropout_masks = []

    for i in range(len(weights)-1):
        Z = np.dot(A, weights[i]) + biases[i]
        A = relu(Z)

        if dropout_rate > 0:
            A, mask = apply_dropout(A, dropout_rate)
            dropout_masks.append(mask)
        else:
            dropout_masks.append(None)

        Zs.append(Z)
        activations.append(A)

    # Output layer
    Z = np.dot(A, weights[-1]) + biases[-1]
    A = sigmoid(Z)

    Zs.append(Z)
    activations.append(A)

    return activations, Zs, dropout_masks

In [ ]:
#loss function
def compute_loss(y, y_hat):
    return -np.mean(y*np.log(y_hat+1e-8) + (1-y)*np.log(1-y_hat+1e-8))

In [ ]:
#back propagation
def backward(y, activations, Zs, weights, dropout_masks, dropout_rate):
    grads_w = []
    grads_b = []

    m = y.shape[0]

    dA = activations[-1] - y

    for i in reversed(range(len(weights))):
        A_prev = activations[i]

        dW = np.dot(A_prev.T, dA) / m
        dB = np.sum(dA, axis=0, keepdims=True) / m

        grads_w.insert(0, dW)
        grads_b.insert(0, dB)

        if i != 0:
            dA = np.dot(dA, weights[i].T)
            dA *= relu_derivative(Zs[i-1])

            if dropout_rate > 0:
                dA *= dropout_masks[i-1]
                dA /= (1 - dropout_rate)

    return grads_w, grads_b

In [ ]:
#sgd
def update_sgd(weights, biases, grads_w, grads_b, lr):
    for i in range(len(weights)):
        weights[i] -= lr * grads_w[i]
        biases[i] -= lr * grads_b[i]
    return weights, biases

In [ ]:
#adam
def init_adam(weights, biases):
    m_w = [np.zeros_like(w) for w in weights]
    v_w = [np.zeros_like(w) for w in weights]
    m_b = [np.zeros_like(b) for b in biases]
    v_b = [np.zeros_like(b) for b in biases]
    return m_w, v_w, m_b, v_b

In [ ]:
def update_adam(weights, biases, grads_w, grads_b, m_w, v_w, m_b, v_b, t, lr):
    beta1, beta2, eps = 0.9, 0.999, 1e-8

    for i in range(len(weights)):
        m_w[i] = beta1*m_w[i] + (1-beta1)*grads_w[i]
        v_w[i] = beta2*v_w[i] + (1-beta2)*(grads_w[i]**2)

        m_b[i] = beta1*m_b[i] + (1-beta1)*grads_b[i]
        v_b[i] = beta2*v_b[i] + (1-beta2)*(grads_b[i]**2)

        m_w_hat = m_w[i] / (1 - beta1**t)
        v_w_hat = v_w[i] / (1 - beta2**t)

        weights[i] -= lr * m_w_hat / (np.sqrt(v_w_hat) + eps)
        biases[i] -= lr * m_b[i] / (np.sqrt(v_b[i]) + eps)

    return weights, biases, m_w, v_w, m_b, v_b

In [ ]:
#training (with early stopping)
def train(X_train, y_train, X_val, y_val, layers, lr=0.01, epochs=100,
          optimizer="sgd", dropout_rate=0.0, patience=10):

    weights, biases = init_params(layers)

    if optimizer == "adam":
        m_w, v_w, m_b, v_b = init_adam(weights, biases)

    best_loss = float('inf')
    patience_counter = 0

    train_losses = []
    val_losses = []

    for epoch in range(epochs):

        # Forward
        activations, Zs, masks = forward(X_train, weights, biases, dropout_rate)
        train_loss = compute_loss(y_train, activations[-1])

        # Backward
        grads_w, grads_b = backward(y_train, activations, Zs, weights, masks, dropout_rate)

        # Update
        if optimizer == "sgd":
            weights, biases = update_sgd(weights, biases, grads_w, grads_b, lr)
        else:
            weights, biases, m_w, v_w, m_b, v_b = update_adam(
                weights, biases, grads_w, grads_b,
                m_w, v_w, m_b, v_b, epoch+1, lr
            )

        # Validation
        val_pred = forward(X_val, weights, biases, 0.0)[0][-1]
        val_loss = compute_loss(y_val, val_pred)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        print(f"Epoch {epoch} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        # Early stopping
        if val_loss < best_loss:
            best_loss = val_loss
            best_weights = [w.copy() for w in weights]
            best_biases = [b.copy() for b in biases]
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping triggered")
            break

    return best_weights, best_biases, train_losses, val_losses

In [ ]:
#prediction
def predict(X, weights, biases):
    probs = forward(X, weights, biases, 0.0)[0][-1]
    return (probs > 0.5).astype(int)

In [ ]:
layers = [X_train.shape[1], 32, 16, 1]

weights, biases, train_losses, val_losses = train(
    X_train, y_train, X_val, y_val,
    layers,
    lr=0.001,
    epochs=100,
    optimizer="adam",
    dropout_rate=0.2,
    patience=10
)

y_pred = predict(X_test, weights, biases)

In [ ]:
# Evaluate the model
print("\n--- Model Evaluation ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()

In [ ]:
#task3:hyperparameter tuning

In [ ]:
#defining search space
learning_rates = [0.001, 0.01, 0.1]

architectures = [
    [X_train.shape[1], 8, 1],
    [X_train.shape[1], 16, 8, 1]
]

optimizers = ["sgd", "adam"]

In [ ]:
#running all combinations
results = []

for lr in learning_rates:
    for arch in architectures:
        for opt in optimizers:

            print(f"\nTraining → LR={lr}, Arch={arch}, Opt={opt}")

            weights, biases, train_losses, val_losses = train(
                X_train, y_train,
                X_val, y_val,
                layers=arch,
                lr=lr,
                epochs=50,
                optimizer=opt,
                dropout_rate=0.2,
                patience=5
            )

            # Validation predictions
            y_val_pred = predict(X_val, weights, biases)

            acc = accuracy_score(y_val, y_val_pred)
            prec = precision_score(y_val, y_val_pred)
            rec = recall_score(y_val, y_val_pred)
            f1 = f1_score(y_val, y_val_pred)

            results.append({
                "Learning Rate": lr,
                "Architecture": str(arch),
                "Optimizer": opt,
                "Accuracy": acc,
                "Precision": prec,
                "Recall": rec,
                "F1 Score": f1,
                "Final Val Loss": val_losses[-1]
            })

In [ ]:
#creating comparison table
results_df = pd.DataFrame(results)

# Sort by best F1 (better than accuracy for imbalance)
results_df = results_df.sort_values(by="F1 Score", ascending=False)

results_df

In [ ]:
#highlighting best model
best = results_df.iloc[0]

print("Best Configuration:\n")
print(best)

In [ ]:
#optimal (visual comparison)
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(results_df["F1 Score"].values, marker='o')
plt.title("F1 Score Comparison")
plt.xlabel("Model Index")
plt.ylabel("F1 Score")
plt.show()

In [ ]:
#task 4: model evaluation

In [ ]:
y_pred = predict(X_test, weights, biases)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

In [ ]:
print(" Model Evaluation:\n")

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print("\nConfusion Matrix:\n", cm)

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(cm)
plt.title("Confusion Matrix")
plt.colorbar()

plt.xlabel("Predicted")
plt.ylabel("Actual")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha='center', va='center')

plt.show()

In [ ]:
#task5: visualisation

# Extract parameters of the best model
best_lr = best["Learning Rate"]
best_arch = eval(best["Architecture"])
best_opt = best["Optimizer"]
best_name = f"LR={best_lr}, Arch={best_arch}, Opt={best_opt}"

print(f"Re-training best model: {best_name}")

# Re-train the best model to get its full training history
best_weights, best_biases, train_losses_best, val_losses_best = train(
    X_train, y_train,
    X_val, y_val,
    layers=best_arch,
    lr=best_lr,
    epochs=100, # Using more epochs to see full training curve
    optimizer=best_opt,
    dropout_rate=0.2, # Assuming dropout rate used in tuning was 0.2
    patience=10 # Assuming patience used in tuning was 10
)

best_history = {
    'train_loss': train_losses_best,
    'val_loss': val_losses_best
}

# Calculate predictions for the best model on the test set
y_pred_best = predict(X_test, best_weights, best_biases)

In [ ]:
# ── Plot 1: Loss vs Epochs for best model ────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(7, 5)) # Changed to single subplot for loss only

ax.plot(best_history['train_loss'], label='Train Loss', linewidth=2)
ax.plot(best_history['val_loss'],   label='Val Loss',   linewidth=2, linestyle='--')
ax.set_title(f'Loss vs Epochs\n({best_name})', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Binary Cross-Entropy Loss')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 2: SGD vs Adam comparison (final val loss for each config) ─────
sgd_results  = results_df[results_df['Optimizer'] == 'sgd']
adam_results = results_df[results_df['Optimizer'] == 'adam']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Accuracy comparison
x = np.arange(len(sgd_results))
width = 0.35
axes[0].bar(x - width/2, sgd_results['Accuracy'].values,  width, label='SGD',  color='steelblue', alpha=0.85)
axes[0].bar(x + width/2, adam_results['Accuracy'].values, width, label='Adam', color='coral',     alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'LR={r}\n{a}' for r, a in zip(sgd_results['Learning Rate'], sgd_results['Architecture'])],
                          rotation=30, fontsize=8)
axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('SGD vs Adam — Test Accuracy', fontweight='bold')
axes[0].axhline(0.8, color='green', linestyle='--', alpha=0.6, label='80% target')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# F1 comparison
axes[1].bar(x - width/2, sgd_results['F1 Score'].values,  width, label='SGD',  color='steelblue', alpha=0.85)
axes[1].bar(x + width/2, adam_results['F1 Score'].values, width, label='Adam', color='coral',     alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'LR={r}\n{a}' for r, a in zip(sgd_results['Learning Rate'], sgd_results['Architecture'])],
                          rotation=30, fontsize=8)
axes[1].set_ylabel('F1 Score')
axes[1].set_title('SGD vs Adam — F1 Score', fontweight='bold')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 3: Learning-rate effect heatmap ─────
pivot_acc = results_df.pivot_table(
    index='Learning Rate', columns=['Optimizer','Architecture'], values='Accuracy')

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(pivot_acc, annot=True, fmt='.3f', cmap='YlGn',
            linewidths=0.5, ax=ax, vmin=0.7, vmax=1.0)
ax.set_title('Test Accuracy Heatmap — LR × Optimizer × Architecture',
             fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Extended model: ReLU + Dropout (0.3) + Early Stopping ─────
print('Training extended ANN: ReLU + Dropout(0.3) + Early Stopping...')
print('='*60)

best_weights_ext, best_biases_ext, train_losses_ext, val_losses_ext = train(
    X_train, y_train, X_val, y_val,
    layers     = [X_train.shape[1], 16, 8, 1],
    lr             = 0.001,
    epochs         = 500,
    optimizer      = 'adam',
    # hidden_act     = 'relu', # This parameter is not part of the `train` function's signature
    dropout_rate   = 0.3,
    patience       = 20
    # verbose        = True # This parameter is not part of the `train` function's signature
)

ext_params = (best_weights_ext, best_biases_ext)
ext_history = {
    'train_loss': train_losses_ext,
    'val_loss': val_losses_ext
}

y_pred_ext = predict(X_test, ext_params[0], ext_params[1])
print('\nExtended Model Metrics:')
print(f'  Accuracy : {accuracy_score(y_test, y_pred_ext):.4f}')
print(f'  Precision: {precision_score(y_test, y_pred_ext):.4f}')
print(f'  Recall   : {recall_score(y_test, y_pred_ext):.4f}')
print(f'  F1-Score : {f1_score(y_test, y_pred_ext):.4f}')

In [ ]:
# ── Loss curve for extended model ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ext_history['train_loss'], label='Train Loss', linewidth=2)
ax.plot(ext_history['val_loss'],   label='Val Loss',   linewidth=2, linestyle='--')
ax.set_title('Extended Model — Loss vs Epochs (with Early Stopping)', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#final comparison ; ANN vs Logistic Regression vs Decision Tree

In [ ]:
# ── Sklearn baseline models (allowed for evaluation/comparison) ──────────
# Use the already-normalised X for LR & DT

# Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train.ravel())
y_pred_lr = lr_model.predict(X_test)

# Decision Tree
dt_model = DecisionTreeClassifier(max_depth=6, random_state=42)
dt_model.fit(X_train, y_train.ravel())
y_pred_dt = dt_model.predict(X_test)

# ── Results table ─────────────────────────────────────────────────────────
comparison = pd.DataFrame({
    'Model': ['ANN (Best Tuned)', 'ANN (Extended: ReLU+Dropout+ES)',
              'Logistic Regression', 'Decision Tree'],
    'Accuracy' : [
        accuracy_score(y_test, y_pred_best),
        accuracy_score(y_test, y_pred_ext),
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_dt)
    ],
    'Precision': [
        precision_score(y_test, y_pred_best),
        precision_score(y_test, y_pred_ext),
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_dt)
    ],
    'Recall': [
        recall_score(y_test, y_pred_best),
        recall_score(y_test, y_pred_ext),
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_dt)
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_best),
        f1_score(y_test, y_pred_ext),
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_dt)
    ]
}).round(4)

print('='*70)
print('FINAL MODEL COMPARISON')
print('='*70)
print(comparison.to_string(index=False))
print('='*70)

In [ ]:
# ── Final comparison bar chart ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
x      = np.arange(len(comparison))
width  = 0.2
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

for i, (metric, color) in enumerate(zip(metrics_list, colors)):
    bars = ax.bar(x + (i - 1.5) * width, comparison[metric],
                  width, label=metric, color=color, alpha=0.85, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(comparison['Model'], rotation=12, ha='right', fontsize=9)
ax.set_ylabel('Score')
ax.set_title('Model Comparison: ANN vs Logistic Regression vs Decision Tree',
             fontweight='bold', fontsize=12)
ax.axhline(0.8, color='black', linestyle='--', alpha=0.5, linewidth=1, label='80% target')
ax.set_ylim(0, 1.1)
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

